# 01 — RAG: Ingestion, Retrieval, Vector Visualization & Cluster Drill-Down
**Goal:** see how different ingestion/chunking choices change retrieval, then *look at* your embedding space instead of trusting it blindly.

# Setup
Run this first in every notebook. It assumes this notebook lives in the same
folder as `inhouse_wrappers.py`, `rag_pure_python.py`, and `inhouse_llm.py`
(the files from earlier in this project). If not, add the folder to `sys.path`.

In [ ]:
import sys, os
# sys.path.append("/path/to/inhouse_rag_capstone")  # uncomment & adjust if needed

from inhouse_llm import (
    multimodal_chat, get_embedding,
    MODEL_QWEN3_14B, MODEL_QWEN3_30B, MODEL_MISTRAL,
    MODEL_LLAMA, MODEL_DEVSTRAL, MODEL_QWEN2_5_VL_7B, MODEL_JINA,
)
from inhouse_wrappers import InHouseLLM, InHouseEmbeddings, llm_for
from rag_pure_python import chunk_text, SimpleVectorStore, generate_answer

print("Setup OK")

## 1. Ingestion — three corpora, three shapes
Why this matters: RAG quality is decided more by *how you ingest* than by which model you call. We'll use three small synthetic 'documents' that stand in for: a plain text doc, a structured/tabular doc, and a FAQ doc.

In [ ]:
docs = {
    "narrative": """
    Retrieval-Augmented Generation (RAG) combines a retriever and a generator.
    The retriever fetches relevant chunks from a vector store using embeddings.
    The generator, an LLM, uses those chunks as context to produce a grounded answer.
    """,
    "tabular": """
    Model: Qwen3-14B | Type: chat | Use: general RAG generation
    Model: Qwen3-32B | Type: chat | Use: harder multi-step reasoning
    Model: Jina-v3 | Type: embedding | Use: retrieval vectors
    Model: Devstral-Small | Type: code | Use: writing MCP tools
    """,
    "faq": """
    Q: What is MCP? A: A protocol that lets an LLM call external tools/data through a standard client-server interface.
    Q: What is an agent? A: An LLM that decides which action to take next in a loop, observes results, and continues.
    Q: What is a vector store? A: A database optimized for nearest-neighbor search over embeddings.
    """,
}
for name, text in docs.items():
    print(name, "->", len(text), "chars")

### Chunking strategy comparison
Fixed-size chunking is naive but fast. For tabular/FAQ-style text, splitting on natural boundaries (newlines, Q/A pairs) preserves meaning better. **When to use which:** fixed-size for unstructured prose; boundary-aware splitting whenever the source has natural delimiters (rows, Q&A, bullet points).

In [ ]:
def chunk_by_lines(text):
    return [line.strip() for line in text.strip().splitlines() if line.strip()]

fixed_chunks = chunk_text(docs["narrative"], chunk_size=80, overlap=10)
line_chunks_tabular = chunk_by_lines(docs["tabular"])
line_chunks_faq = chunk_by_lines(docs["faq"])

print("Fixed-size chunks (narrative):", len(fixed_chunks))
for c in fixed_chunks: print(" -", c[:60])
print("\nLine chunks (tabular):", len(line_chunks_tabular))
print("\nLine chunks (FAQ):", len(line_chunks_faq))

## 2. Build the index
All chunks go into one `SimpleVectorStore` regardless of source — this mimics a real pipeline where ingestion is per-source but the index is unified.

In [ ]:
all_chunks = fixed_chunks + line_chunks_tabular + line_chunks_faq
store = SimpleVectorStore(embedding_model=MODEL_JINA)
store.add(all_chunks)
print(f"Indexed {len(store.texts)} chunks")

## 3. Retrieval — compare top-k for different queries
Try a query that should hit the FAQ chunks vs one that should hit the tabular chunks. **Why look at this:** retrieval failures are silent — the LLM will confidently answer from the wrong chunk if retrieval picked the wrong one. Always eyeball top-k during dev.

In [ ]:
for q in ["What is MCP?", "Which model should I use for code?"]:
    print("Query:", q)
    for text, score in store.search(q, k=3):
        print(f"  [{score:.3f}] {text[:70]}")
    print()

## 4. Visualizing the embedding space
Embeddings live in high dimensions (hundreds+). We reduce to 2D with PCA (fast, linear, preserves global structure) to *see* whether semantically similar chunks actually cluster together. If they don't, that's a signal your embedding model or chunking isn't capturing what you think it is.

`pip install scikit-learn matplotlib --break-system-packages` if missing.

In [ ]:
import numpy as np
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

vectors = np.array(store.vectors)
labels = (["narrative"] * len(fixed_chunks) +
          ["tabular"] * len(line_chunks_tabular) +
          ["faq"] * len(line_chunks_faq))

coords = PCA(n_components=2).fit_transform(vectors)

plt.figure(figsize=(7, 5))
for label in set(labels):
    idx = [i for i, l in enumerate(labels) if l == label]
    plt.scatter(coords[idx, 0], coords[idx, 1], label=label)
plt.legend()
plt.title("Chunk embeddings (PCA, colored by source type)")
plt.show()

## 5. Cluster drill-down
KMeans groups chunks by embedding similarity *without* using our source labels. Compare the unsupervised clusters to the source-type colors above — do they roughly agree? Then drill into one cluster to read what's actually in it.

**Why this matters for a capstone:** clustering retrieved/indexed chunks is how you spot duplicate content, mixed-topic chunks (bad splits), or an under-represented topic in your knowledge base before it causes bad answers in production.

In [ ]:
from sklearn.cluster import KMeans

k = 3
kmeans = KMeans(n_clusters=k, random_state=0, n_init=10).fit(vectors)
cluster_ids = kmeans.labels_

plt.figure(figsize=(7, 5))
plt.scatter(coords[:, 0], coords[:, 1], c=cluster_ids, cmap="tab10")
plt.title(f"Same chunks, colored by KMeans cluster (k={k})")
plt.show()

# Drill into cluster 0
print("--- Contents of cluster 0 ---")
for text, cid in zip(all_chunks, cluster_ids):
    if cid == 0:
        print(" -", text[:80])

### Try this
- Increase `k` and see if clusters start splitting the FAQ into finer sub-topics.
- Add a 4th, deliberately off-topic document and confirm it forms its own cluster — that's your sanity check that the pipeline can detect irrelevant content.